# Week 8, day 1 -- RAG over tasting notes

Retrieval asks a different question from the models so far: not "what does this prose imply about
price" but "what did wines that taste like this actually cost".

The store holds tasting notes only. No price, no critic score, no winery in the embedded text -- if
those went in, similarity search would find the answer instead of a comparable wine, and the whole
evaluation would be a leak with extra steps. Prices live in the metadata, retrieved *after* the
match, which is how a comparable is supposed to work.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE

from pricer import vectors
from pricer.agents import ClassicalAgent, FrontierAgent, NeighboursAgent, setup_logging
from pricer.evaluator import Report, leaderboard
from pricer.items import Wine

setup_logging()
train, val, test = Wine.load_local()
collection = vectors.load()  # built by scripts/vectors.py
encoder = vectors.encoder()
print(f"{collection.count():,} tasting notes embedded")

### Does the embedding space know about price?

If wines that taste alike also cost alike, the neighbourhood structure carries price information and
retrieval will help. Colour a t-SNE projection by price and look for gradient rather than noise.

In [ ]:
embeddings, prices, varieties = vectors.sample_coordinates(collection, limit=2_000)
flat = TSNE(n_components=2, random_state=42, init="pca", perplexity=30).fit_transform(embeddings)
plt.figure(figsize=(9, 7))
points = plt.scatter(flat[:, 0], flat[:, 1], c=np.log1p(prices), cmap="RdYlGn_r", s=8)
plt.colorbar(points, label="log1p(price)")
plt.title("2,000 tasting notes, coloured by price")
plt.show()
print("Most common varieties in the sample:", Counter(varieties).most_common(5))

### What does retrieval return for one wine?

In [ ]:
wine = test[7]
notes, found = vectors.similar(wine.description, collection, encoder, k=5)
print(f"{wine.label} -- actually ${wine.price:.0f}\n")
for note, price in zip(notes, found, strict=True):
    print(f"${price:>6.0f}  {note[:110]}...")
print(f"\ngeometric mean of the neighbours: ${np.expm1(np.log1p(found).mean()):.2f}")

### Retrieval alone, then retrieval plus a language model

Two agents, one question each. `NeighboursAgent` is pure retrieval: the geometric mean of the k
nearest prices, no LLM. `FrontierAgent` puts the same neighbours in a prompt and asks a model for a
number. The gap between them is what the language model contributes over the lookup; if it is small,
the expensive part is not earning its keep.

100 test wines, because the frontier agent goes over the network for each one.

In [ ]:
sample = test[:100]
neighbours = NeighboursAgent(collection, encoder)
classical = ClassicalAgent()

for name, agent in [("Neighbours (k=8, retrieval only)", neighbours), ("Classical (TF-IDF + Ridge)", classical)]:
    guesses = [agent.price(w.description) for w in sample]
    Report(name, [w.label for w in sample], guesses, [w.price for w in sample]).save()

frontier = FrontierAgent(collection, encoder)
guesses = [frontier.price(w.description) for w in sample]
Report("Frontier (RAG + LLM)", [w.label for w in sample], guesses, [w.price for w in sample]).save()
leaderboard()

### Experiments worth running here

- Sweep `k` in `NeighboursAgent`. Too few neighbours is noisy, too many regresses to the mean.
- Weight the neighbours by similarity instead of averaging them flat.
- Retrieve on the LLM summary instead of the full note (`scripts/tasting.py` first) and see whether a
  tighter, more structured text retrieves better comparables.
- Embed with a bigger encoder (`all-mpnet-base-v2`) and measure whether the extra dimensions pay.
- Give the frontier agent the neighbours' varieties and regions too, and see if context helps or
  just distracts it.